# Selection Analysis

Everything about **what the truth contains and what the selections keep** --
kept deliberately separate from
`Evaluation_ChargeLightMatching_BeforeBeamWindowCut.ipynb`, which owns
**reconstruction performance** (clustering/imaging detail, completeness, purity,
true-reco matching).

Reads the charge-light-matching files/events exactly the way the evaluation
notebook does (same `PARENT_DIR`, same file/event discovery, same `target_file` /
`target_event` / `target_file_range` selectors, same selection parameters, same
cut functions -- no cut is reimplemented here).

## What this notebook produces

**1. Selection flow** -- how many clusters survive each successive cut, drawn as
horizontal bar blocks, one block per stage.

*True side* (truth available, `q_true>0` = neutrino), three separate bars per
block -- cosmic CLUSTERS, and neutrino INTERACTIONS from `mc.json` drawn as two
bars of their own, vertex in-volume and vertex out-of-volume (side by side, not
stacked into one neutrino bar, so each length is read straight off the axis):

1. No cuts
2. True Deposited energy >= `min_cluster_energy` (100 MeV)

*Reco side* -- one bar per block (Total only; reco clusters carry no truth label,
so there is no cosmic/neutrino split to draw):

1. No cuts
2. + beam window

There is no true-side beam-window stage: "in beam window" is not a truth
quantity -- true clusters carry no flash and no time, so the label could only be
inferred by matching to a beam-window-flashed reco cluster, mixing beam timing
with reconstruction completeness. Beam window is measured where the flash is, on
the reco side.

There is no dead-area stage on either side. On reco it would be meaningless --
points are never reconstructed inside a dead area, so there is nothing to remove.
On true it IS applied (it removes deposits the detector could never have seen, so
that true and reco are compared over the same measurable volume) but it removes
no whole cluster, so it is not a stage of the flow; its effect shows up in the
cluster energies and point counts reported below.

Counts go to `selection_flow_counts.txt` and `summary.txt`; the drawing lives in
`DrawRecoTrueClusterCount.py`, the counting in the main loop next to the cuts it
is counting.

**2. Job-level neutrino selection plots and info files**, in `job_summary/`:

| Output | What it shows |
|---|---|
| `labels_aggregated_job_*.png` | neutrino vs cosmic true cluster counts |
| `labels_by_nu_idx_job_*.png` | true clusters per neutrino interaction index |
| `true_neutrino_vertices_job_*.png` | interaction vertex positions |
| `true_neutrino_vertex_volume_job_*.png` | vertices in vs out of the sensitive volume |
| `true_neutrino_flavor_in_volume_job_*.png` | flavor composition of in-volume interactions |
| `true_neutrino_breakdown_job_*.png` | all interactions -> those that deposited -> those surviving the cuts |
| `true_neutrino_info.txt` | one row per interaction: vertex, volume flag, energies |
| `removed_true_neutrino_info.txt` | the interactions the cuts removed, and which cut did it |

These ran at job level in the evaluation notebook until the two analyses were
split. The functions are unchanged and still shared -- the evaluation notebook
keeps drawing its own **event- and file-level** copies of the six plots for
debugging individual events and files, and keeps
`cosmic_clusters_by_category_*` at all levels (that one classifies cosmic track
geometry via `cluster_category`, which is cluster categorisation rather than
neutrino selection).

Cuts are cumulative: every stage is applied on top of the clusters that survived
the stage above it. The cosmic bar is labelled with its raw cluster count; each
neutrino bar is labelled with its interaction count and its share of THAT
stage's neutrino total, so the two neutrino shares sum to 100% and the
in/out-of-volume composition shift from one stage to the next is readable
directly. Stage-to-stage survival percentages are in
`selection_flow_counts.txt`.

In [17]:
# Charge-light matching is a combined-APA evaluation (img-global / sed-sce are
# already global across APAs) -- no per-APA/face looping like the older pipeline.
files     = "all"   # "all", or 1/2/3/... to limit the number of file subdirectories processed
events    = "all"   # "all", or 1/2/3/... to limit the number of events processed per file

# ========================================================================
# SELECTIVE FILE/EVENT FILTERING (Optional)
# ========================================================================
# Set to None to process all files/events, or specify to run only specific ones
# Example: target_file = "file0", target_event = 3  (to process only file0, event 3)
target_file  = None  # Set to "file0", "file1", etc. to process specific file only
target_event = None   # Set to event number (0, 1, ..., 9) to process specific event only

# Range of file INDICES to run, inclusive on both ends: (6, 9) runs file6, file7,
# file8, file9. None runs every file. Matched on the number at the end of the
# directory name, NOT on position in the list -- the directories sort
# lexicographically (file0, file1, file10, file11, file2, ...), so a positional
# slice would pick the wrong files. This is the only file selector that survives
# that sort order; the `files = N` knob above still takes the first N in
# lexicographic order.
# NOTE: target_file (above) is applied too, so set it to None when using a range,
# otherwise only the one file that satisfies both runs.
target_file_range = None   # e.g. (6, 9) for file6..file9


# ========================================================================
# Decide which plots to draw
# ========================================================================
b_draw_event_level_plots = False   # Draw event-level plots (one per event)
b_draw_file_level_plots  = True   # Draw file-level plots (one per file)
b_draw_job_level_plots   = True   # Draw job-level plots (one per job)

In [18]:
%load_ext autoreload
%autoreload 2

# python libraries
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from pathlib import Path
import sys
import os
from scipy.spatial import KDTree
import pandas as pd
import seaborn as sns
import time
import io
import contextlib
from datetime import datetime, timedelta

np.set_printoptions(linewidth=1000)

# Record job start time (used to report total job runtime at the end)
job_start_time = time.time()
print(f"Job started at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Job started at: 2026-08-05 16:38:12


In [19]:
# Import functions from Python modules.
# Same reading/selection functions as Evaluation_ChargeLightMatching_BeforeBeamWindowCut.ipynb
# (identical cut implementations -- this notebook only counts what survives them,
# it does not redefine any cut), plus the selection-flow drawers.
#
# The neutrino-selection drawers/writers at the bottom of this list used to run at
# JOB level in the evaluation notebook; they live here now. The evaluation notebook
# still draws its own EVENT- and FILE-level copies of the same six drawers for
# debugging -- same functions, called from both notebooks, defined once.
from readfiles import ensure_data_extracted, read_charge_light_files_for_event, flatten_mc_tree
from selections import (
    GroupClustersByID, build_true_points_charge_light, apply_deadarea_cut_true_charge_light,
    reassign_cluster_ID_true_charge_light,
    apply_energy_cutoff, apply_min_true_points_cutoff,
    apply_wire_readout_sensitive_yz_plane_cut_true, apply_wire_readout_sensitive_yz_plane_cut_reco,
)
from metadata import (
    build_cluster_flash_metadata, build_img_cluster_flash_metadata,
    build_true_cluster_type_records, build_neutrino_vertex_records,
)
from DrawRecoTrueFlashes import BEAM_WINDOW_MIN_US, BEAM_WINDOW_MAX_US
from DrawRecoTrueClusterCount import (
    DrawTrueClusterSelectionFlow, DrawRecoClusterSelectionFlow, write_selection_flow_table,
)
# Job-level neutrino selection plots (moved here from the evaluation notebook).
from DrawRecoTrueClusters import (
    DrawLabelsAggregated, DrawLabelsByNuIdx,
    DrawNeutrinoVertices, DrawNeutrinoVolumeCategory, DrawNeutrinoFlavor, DrawNeutrinoBreakdown,
)
# Information-file writers live in writeinformation.py, not metadata.py: metadata
# builds the in-memory records, this writes the human-readable .txt tables.
from writeinformation import write_neutrino_vertex_info, write_removed_neutrino_info

In [20]:
# Configuration: Parent directory containing multiple file subdirectories (file0/, file1/, ...)
#
# Expected structure (per file subdirectory). The preprocessed tree has no zip --
# its data/ is already present, so ensure_data_extracted() below simply no-ops.
# PARENT_DIR/
#   file0/data/0/0-img-global.json                      (reco clusters, combined APA)
#   file0/data/0/0-sed-sce_drift_smear_readout.json      (true clusters, combined APA)
#   file0/data/0/0-mc.json                               (particle truth ancestry tree)
#   file0/data/0/0-op.json                               (optical/light info)
#   file0/data/1/, 2/, ... (one subdirectory per event)
#   file1/mabc.zip, file1/data/..., etc.

# The DEAD-AREA-PREPROCESSED tree, produced once by preprocess_deadarea_cut.py.
# Its true-point files already have the dead-area cut applied, which is why
# Apply_deadarea_cut is False below -- see the note there. Read that script's
# docstring, or DEADAREA_PREPROCESSING.txt inside the tree, before switching this
# back to the raw tree: the two are NOT interchangeable.
PARENT_DIR = Path("Haiwang_files_charge_light_matching_MCP2025C_Fall_production_after_deadareacut")

# Number of files to process (convert 'files' variable to num_files_to_process)
num_files_to_process = None if files == "all" else files

# Number of events to process (convert 'events' variable to num_events_to_process)
num_events_to_process = None if events == "all" else events

# Output directory for plots
PLOTBASEDIR = Path("multi_file_plots_charge_light_matching")
PLOTBASEDIR.mkdir(parents=True, exist_ok=True)

print("Configuration:")
print(f"Parent directory: {PARENT_DIR}")
print(f"Plot base directory: {PLOTBASEDIR}")
print(f"Files to process: {files}")
print(f"Events to process: {events}")

if target_file is not None or target_event is not None or target_file_range is not None:
    print(f"\n⚡ SELECTIVE FILTERING ENABLED:")
    print(f"  Target file: {target_file if target_file else 'all'}")
    print(f"  Target event: {target_event if target_event is not None else 'all'}")
    if target_file_range is not None:
        print(f"  Target file range: file{target_file_range[0]}..file{target_file_range[1]} (inclusive)")

# ========================================================================
# SELECTION PARAMETERS
# ========================================================================
# Matching/completeness/purity radii and the geometry-based cuts (fiducial YZ box,
# dead-area) are reused at the SAME values as the existing pipeline -- detector
# geometry hasn't changed and these are unit-independent of the new q/energy field.
radius_completeness         = 2
radius_purity_xz          = 2
radius_purity_yz          = 5
radius_purity_xy          = 5
min_recopoints_threshold  = 5

# min_true_points_cutoff / min_reco_points_cutoff are DISABLED for now: this
# format's point clouds are much sparser than the old imaging-based
# reconstruction -- real neutrino clusters have been seen with as few as 13
# points -- so the old threshold (200) would delete real signal clusters
# outright. Revisit once correct values are known for this format's point
# density.
#
# min_cluster_energy IS applied (Apply_energy_cutoff = True): sed-sce's
# per-point 'e' field (MeV) is a genuine energy deposit -- same physical
# quantity/units as the old (non charge-light) pipeline's energy column --
# so the old threshold (100 MeV) carries over directly. See
# build_true_points_charge_light's energy= parameter (falls back to 'q'
# for older-format files that lack 'e').
min_cluster_energy        = 100     # APPLIED (Apply_energy_cutoff = True below)
min_true_points_cutoff    = 200     # NOT APPLIED (Apply_min_true_points_cutoff = False below)
min_reco_points_cutoff    = 200     # NOT APPLIED (Apply_min_reco_points_cutoff = False below)

# Apply selections
Apply_energy_cutoff                         = True    # sed-sce's 'e' field is genuine MeV -- see note above
Apply_min_true_points_cutoff                = False   # disabled -- see note above
Apply_min_reco_points_cutoff                = False   # disabled -- see note above
Apply_wire_readout_sensitive_xz_plane_cut   = True
Apply_time_window_cut                       = False   # must stay disabled -- no per-point true time in this format
# The dead-area cut is APPLIED, just not here: PARENT_DIR above is the tree
# preprocess_deadarea_cut.py already cut. Applying it again would be a no-op that
# costs a polygon test per event. It is applied FIRST, before the energy cut,
# because reco points are never reconstructed inside a dead channel region -- true
# deposits there could never have been seen, so removing them is a correction that
# puts truth and reco on the same measurable volume, not a selection to rank
# alongside the others. Set this True only if you point PARENT_DIR back at a raw
# tree, and note that doing so restores the OLD ordering (dead area last).
Apply_deadarea_cut                          = False

# YZ Sensitivity cut parameters (detector geometry, unit: cm)
x_min = -250.0
x_max = 250.0
y_min = -200.0
y_max = 200.0
z_min = 0.15
z_max = 500.85

# Metadata label only (add_metadata_true_clusters/add_metadata_true_reco_pair_cluster
# store 'view' as a plain string field, not used for any logic) -- there's no
# 2-view/3-view distinction in the charge-light format, so this is just a constant.
view = "combined"

marker_size = 1

print("\nCuts applied:")
if Apply_energy_cutoff:
    print(f"- Energy cutoff applied (threshold {min_cluster_energy} MeV, using sed-sce's per-point 'e' field)")
if Apply_wire_readout_sensitive_xz_plane_cut:
    print(f"- Wire readout sensitive xz plane cut applied")
if Apply_deadarea_cut:
    print(f"- Dead area cut applied HERE (split by X sign into APA0/APA1, see apply_deadarea_cut_true_charge_light)")
else:
    print(f"- Dead area cut applied UPSTREAM by preprocess_deadarea_cut.py (baked into PARENT_DIR, before every other cut)")
print("\nCuts not applied (see note above)")
if not Apply_min_true_points_cutoff:
    print(f"- Minimum true points cutoff not applied (threshold {min_true_points_cutoff} too aggressive for this format's point density)")
if not Apply_min_reco_points_cutoff:
    print(f"- Minimum reco points cutoff not applied (threshold {min_reco_points_cutoff} too aggressive for this format's point density)")

# ========================================================================
# ONE-TIME EXTRACTION
# ========================================================================
# Each file subdirectory ships as a single zip file. ensure_data_extracted()
# only unzips if that file's data/ folder doesn't already exist yet, so
# re-running this notebook never re-extracts.
if PARENT_DIR.exists():
    for subdir in sorted(PARENT_DIR.iterdir()):
        if subdir.is_dir():
            ensure_data_extracted(subdir)
else:
    print(f"Error: Parent directory {PARENT_DIR} does not exist")


Configuration:
Parent directory: Haiwang_files_charge_light_matching_MCP2025C_Fall_production_after_deadareacut
Plot base directory: multi_file_plots_charge_light_matching
Files to process: all
Events to process: all

Cuts applied:
- Energy cutoff applied (threshold 100 MeV, using sed-sce's per-point 'e' field)
- Wire readout sensitive xz plane cut applied
- Dead area cut applied UPSTREAM by preprocess_deadarea_cut.py (baked into PARENT_DIR, before every other cut)

Cuts not applied (see note above)
- Minimum true points cutoff not applied (threshold 200 too aggressive for this format's point density)
- Minimum reco points cutoff not applied (threshold 200 too aggressive for this format's point density)


In [21]:
def find_all_input_directories(parent_dir):
    """
    Scan parent directory for all subdirectories containing 'data' folder.
    Returns a list of file directories (file0/, file1/, etc.).
    """
    parent_dir = Path(parent_dir)
    if not parent_dir.exists():
        print(f"Error: Parent directory {parent_dir} does not exist")
        return []

    data_dirs = []
    for subdir in sorted(parent_dir.iterdir()):
        if subdir.is_dir():
            data_path = subdir / "data"
            if data_path.exists() and data_path.is_dir():
                data_dirs.append(subdir)
                print(f"Found: {subdir}")

    return data_dirs

def file_index_from_name(name):
    """
    Trailing integer of a file directory name ("file10" -> 10), or None if it
    has no trailing digits. Used by target_file_range so files are selected by
    their real index rather than by position in the lexicographically sorted
    list (file0, file1, file10, file11, file2, ...).
    """
    digits = ""
    for ch in reversed(name):
        if not ch.isdigit():
            break
        digits = ch + digits
    return int(digits) if digits else None


def detect_events_in_directory(input_dir):
    """
    Auto-detect the number of events in a directory.
    Events are identified as numeric subdirectories in data/.
    Returns a sorted list of event numbers.
    """
    input_dir = Path(input_dir)
    data_dir = input_dir / "data"

    if not data_dir.exists():
        print(f"Warning: Data directory {data_dir} does not exist")
        return []

    events = []
    for item in data_dir.iterdir():
        if item.is_dir():
            try:
                event_num = int(item.name)
                events.append(event_num)
            except ValueError:
                pass

    return sorted(events)

# Auto-detect all input directories from parent directory
print(f"Scanning parent directory: {PARENT_DIR}")
print("-" * 60)
input_directories = find_all_input_directories(PARENT_DIR)
# Limit to num_files_to_process files for testing
if num_files_to_process is not None:
    input_directories = input_directories[:num_files_to_process]
else:
    num_files_to_process = len(input_directories)
print("-" * 60)

print(f"\nFound {len(input_directories)} input directories with data/\n")
if input_directories:
    for input_dir in input_directories:
        detected_events = detect_events_in_directory(input_dir)
        if detected_events:
            print(f"  {input_dir.name}/data/: {len(detected_events)} events ({min(detected_events)}-{max(detected_events)})")
        else:
            print(f"  {input_dir.name}/data/: No events found")
else:
    print(f"Error: No subdirectories with 'data/' found in {PARENT_DIR}")


Scanning parent directory: Haiwang_files_charge_light_matching_MCP2025C_Fall_production_after_deadareacut
------------------------------------------------------------
Found: Haiwang_files_charge_light_matching_MCP2025C_Fall_production_after_deadareacut/file0
Found: Haiwang_files_charge_light_matching_MCP2025C_Fall_production_after_deadareacut/file1
Found: Haiwang_files_charge_light_matching_MCP2025C_Fall_production_after_deadareacut/file10
Found: Haiwang_files_charge_light_matching_MCP2025C_Fall_production_after_deadareacut/file11
Found: Haiwang_files_charge_light_matching_MCP2025C_Fall_production_after_deadareacut/file2
Found: Haiwang_files_charge_light_matching_MCP2025C_Fall_production_after_deadareacut/file3
Found: Haiwang_files_charge_light_matching_MCP2025C_Fall_production_after_deadareacut/file4
Found: Haiwang_files_charge_light_matching_MCP2025C_Fall_production_after_deadareacut/file5
Found: Haiwang_files_charge_light_matching_MCP2025C_Fall_production_after_deadareacut/file6
Fou

In [22]:
# ============================================================================
# MAIN LOOP -- selection flow counts + the job-level neutrino selection records
# ============================================================================
# Two things are produced per event:
#
#   1. SELECTION FLOW counts -- how many clusters survive each successive cut.
#   2. The RECORDS behind the job-level neutrino selection plots (cluster-type
#      records and mc.json vertex records), which used to be built in the
#      evaluation notebook and are built here now.
#
# Both come off the same cut chain, applied in the main pipeline's order with the
# main pipeline's own functions and the parameters from the config cell above --
# nothing is reimplemented here:
#
#   energy (min_cluster_energy) -> min true points -> wire-readout sensitive
#   YZ box -> dead area
#
# The FLOW snapshots ids after the energy cut; the RECORDS use the fully cut
# clusters, exactly as the evaluation notebook does, so the plots drawn here match
# the event/file-level copies it still draws.
#
# The flow's true side STARTS from the energy cut rather than from the raw uncut
# clusters: uncut, an event carries ~227 true clusters, almost all of them tiny
# sub-threshold depositions, which swamps every later stage (20356 clusters
# before the cut vs 76 after the beam window). Starting at the energy cut makes
# the first block the same population near_miss_investigation.py reports, so the
# two analyses' baselines agree.
#
# "Beam window" is a RECO-side quantity only: a reco cluster is in the beam window
# if its associated flash time falls inside [BEAM_WINDOW_MIN_US, BEAM_WINDOW_MAX_US].
# There is deliberately no true-side beam-window stage -- true clusters carry no
# flash and no time, so the label could only be inferred by matching to a
# beam-window-flashed reco cluster, mixing beam timing with reconstruction
# completeness while reading as a truth-level selection. Reco clusters are grouped by
# real_cluster_id (not cluster_id, which can merge physically distinct tracks --
# see the evaluation notebook's header note).
#
# NO DEAD-AREA CUT ON THE RECO SIDE. Reco points cannot be reconstructed inside a
# dead area in the first place, so there is nothing there for the cut to remove --
# applying it to reco would be a guaranteed no-op dressed up as a selection. The
# cut is a true-side correction: it removes true deposits that the detector could
# never have seen, so that true and reco are compared over the same measurable
# volume. (An earlier version of this notebook claimed to apply it to reco; it
# never did, and it should not.)
#
# The cut functions print per-cluster detail (hundreds of lines per event, ~53k
# lines over a full job), so their stdout is suppressed -- the counts are what
# matter here. The dead-area cut additionally takes verbose=False, which skips
# building the per-cluster tally those prints need; that tally is
# O(n_clusters x n_points) and is the dominant cost of the cut once nobody is
# reading the output. Neither changes the filtering.

# One directory per run, all of them under a single selection_analysis/ parent:
# selection_analysis/<YYYYmmdd_HHMMSS>/. The timestamp is the run's start time, so
# runs never overwrite each other and sort chronologically. A run worth keeping can
# be renamed by hand with a descriptor appended after the timestamp
# (20260802_145208_allfiles_allevents_NewProcedure) -- nothing below reads the
# directory name back, so renaming it is safe.
timestamp  = datetime.now().strftime("%Y%m%d_%H%M%S")
output_dir = PLOTBASEDIR / "selection_analysis" / timestamp
output_dir.mkdir(parents=True, exist_ok=True)
print(f"\n{'='*70}")
print(f"Output directory: {output_dir}")
print(f"{'='*70}\n")

# Stage definitions: (key, label, is_geometry_stage). is_geometry_stage marks the
# stages dropped from the "without geometry cuts" version of each plot.
# The dead-area / wire-readout stage is not tracked as a stage of its own: measured
# over the full dataset it removed no whole CLUSTER from either side, so it only
# added a redundant row and a redundant block. It is still APPLIED to the true side
# (it changes point counts and cluster energies, which the records below report).
TRUE_STAGES = [
    ('nocut',    'No cuts',                                   False),
    ('energy',   f'True Deposited energy >= {min_cluster_energy} MeV',   False),
]
RECO_STAGES = [
    ('nocut',    'No cuts',                                   False),
    ('beam',     '+ Beam window',                             False),
]

# Neutrinos are counted as mc.json INTERACTIONS split by vertex volume, not as
# clusters: at the no-cut stage an interaction that deposited nothing has no
# cluster to count, so a cluster-based tally would silently omit it (94 of 223
# over the full dataset). Cosmics stay cluster counts -- they have no mc.json
# vertex and no interaction concept.
job_true_counts = {key: {'cosmic': 0, 'neutrino_in': 0, 'neutrino_out': 0} for key, _, _ in TRUE_STAGES}
job_reco_counts = {key: {'total': 0} for key, _, _ in RECO_STAGES}

# Job-level accumulators for the moved neutrino selection plots.
job_cluster_type_records = []     # per-true-cluster is_neutrino records (build_true_cluster_type_records)
job_vertex_records       = []     # per true neutrino interaction (build_neutrino_vertex_records)

total_events_processed = 0
total_files_processed  = 0


def surviving_ids(points):
    """Distinct cluster_ids (column 3) left in a point array; empty set if none."""
    if points is None or len(points) == 0:
        return set()
    return set(np.unique(np.asarray(points)[:, 3]))


def points_of(clusters, keep_ids):
    """Stack the points of the clusters whose id is in keep_ids (empty -> None)."""
    kept = [np.asarray(pts) for cid, pts in clusters.items() if cid in keep_ids]
    return np.vstack(kept) if kept else None


for file_idx, input_dir in enumerate(input_directories):
    input_file_name = input_dir.name

    # SELECTIVE FILTERING: same selectors as the main notebook
    if target_file is not None and input_file_name != target_file:
        print(f"Skipping {input_file_name} (target: {target_file})")
        continue
    if target_file_range is not None:
        file_idx_parsed = file_index_from_name(input_file_name)
        range_low, range_high = target_file_range
        if file_idx_parsed is None or not (range_low <= file_idx_parsed <= range_high):
            print(f"Skipping {input_file_name} (target range: file{range_low}..file{range_high})")
            continue

    print(f"\n{'='*70}")
    print(f"FILE {file_idx+1}/{len(input_directories)}: {input_dir}")
    print(f"{'='*70}")

    events_list = detect_events_in_directory(input_dir)
    if not events_list:
        print(f"No events found in {input_dir}, skipping...")
        continue

    event_low = min(events_list)
    event_high = max(events_list) + 1 if num_events_to_process is None else event_low + num_events_to_process
    total_files_processed += 1

    for evt in range(event_low, event_high):
        if target_event is not None and evt != target_event:
            continue

        result = read_charge_light_files_for_event(input_dir, evt)
        if result is None:
            print(f"  Event {evt}: could not read data, skipping")
            continue
        event_key = f"{input_file_name}_{evt}"

        # ------------------------------------------------------------------
        # BEAM-WINDOW CLUSTERS (reco side), via the flash bridge
        # ------------------------------------------------------------------
        event_flash_metadata_list = build_cluster_flash_metadata(
            result['op'], input_file_name, evt, "Combined", event_key)
        event_img_cluster_flash_records = build_img_cluster_flash_metadata(
            result['reco'], result['clustering'], event_flash_metadata_list,
            input_file_name, evt, "Combined", event_key)

        x_clu, y_clu, z_clu, id_clu, q_clu, real_id_clu = result['clustering']
        reco_points_all  = np.column_stack((x_clu, y_clu, z_clu, real_id_clu, q_clu))
        clusters_all_clu = GroupClustersByID(reco_points_all)

        clu_beam_window_ids = {float(r['clustering_cluster_id']) for r in event_img_cluster_flash_records
                               if BEAM_WINDOW_MIN_US <= r['flash_time'] <= BEAM_WINDOW_MAX_US}
        clusters_clu_in_beam_window = {cid: pts for cid, pts in clusters_all_clu.items()
                                       if cid in clu_beam_window_ids}

        # ------------------------------------------------------------------
        # TRUE SIDE: build + reassign IDs (no cuts yet), then cut cumulatively
        # ------------------------------------------------------------------
        x_true, y_true, z_true, id_true, q_true, real_id_true, e_true, nu_idx_true = result['true_clustering']
        true_points_all = build_true_points_charge_light(
            # real_id_true (real_cluster_id), NOT id_true (cluster_id): cluster_id is a
            # coarser grouping that can merge physically distinct tracks, real_cluster_id
            # is the per-track ID (same convention as the reco side).
            x_true, y_true, z_true, real_id_true, q_true, energy=e_true, nu_idx=nu_idx_true)
        true_points_all = reassign_cluster_ID_true_charge_light(true_points_all)

        # Snapshot BEFORE the cuts, grouping only. Two jobs: the flow's "No cuts"
        # stage, and build_neutrino_vertex_records' removal diagnostics -- it uses
        # this to say what a removed neutrino actually deposited, and hence which
        # cut removed it (removed_true_neutrino_info.txt).
        clusters_true_precut = GroupClustersByID(true_points_all)

        # cluster_id -> is_neutrino, fixed once before any cut: IDs are assigned
        # by reassign_cluster_ID_true_charge_light BEFORE the cuts and the cuts
        # only ever drop points/clusters, never renumber, so this stays valid at
        # every stage below.
        is_neutrino_by_cid = {cid: bool(np.asarray(pts)[0, 4] > 0)
                              for cid, pts in clusters_true_precut.items()}

        true_stage_ids = {'nocut': set(clusters_true_precut.keys())}

        # The cut chain, in the evaluation notebook's order. stdout suppressed
        # (see header note); a fresh StringIO per event so nothing accumulates.
        true_points = true_points_all
        with contextlib.redirect_stdout(io.StringIO()):
            if Apply_energy_cutoff:
                true_points = apply_energy_cutoff(true_points, min_cluster_energy)
            # Flow snapshot: taken here, so the flow's energy stage is unaffected by
            # the geometry cuts that follow (they remove points, not whole clusters).
            true_stage_ids['energy'] = surviving_ids(true_points)

            if Apply_min_true_points_cutoff:
                true_points = apply_min_true_points_cutoff(true_points, min_true_points_cutoff)
            if Apply_wire_readout_sensitive_xz_plane_cut:
                true_points = apply_wire_readout_sensitive_yz_plane_cut_true(
                    true_points, x_min, x_max, y_min, y_max, z_min, z_max)
            if Apply_deadarea_cut:
                # output_dir=None: no before/after plot. verbose=False: skip the
                # per-cluster tally that only feeds the prints (see header note).
                true_points = apply_deadarea_cut_true_charge_light(
                    true_points, output_dir=None, event=evt, file_name=input_file_name,
                    verbose=False)

        # Unlike the evaluation notebook, an event with nothing left is NOT skipped:
        # a fully-cut event is a real data point for a selection flow, and its
        # neutrino interactions still belong in removed_true_neutrino_info.txt.
        clusters_true = GroupClustersByID(true_points) if len(true_points) > 0 else {}

        # ------------------------------------------------------------------
        # NEUTRINO/COSMIC LABEL RECORDS (per true cluster) -- feeds
        # DrawLabelsAggregated / DrawLabelsByNuIdx at job level below.
        # ------------------------------------------------------------------
        event_cluster_type_records = build_true_cluster_type_records(
            clusters_true, input_file_name, evt, event_key)

        # ------------------------------------------------------------------
        # NEUTRINO INTERACTIONS (mc.json), joined to their true cluster by nu_idx
        # (cluster_id = 99990+nu_idx -- an exact key, no spatial matching).
        #
        # ONE call serves both consumers. The flow loop below reads only
        # 'cluster_id' and 'vertex_in_volume', and build_neutrino_vertex_records
        # emits one record per mc.json interaction with both fields set
        # independently of which clusters dict it is given -- so passing the
        # POST-cut clusters (as the evaluation notebook does, which is what makes
        # the plots here match its event/file-level copies) leaves the flow counts
        # untouched. clusters_true_precut + min_cluster_energy populate
        # removal_reason, which is what removed_true_neutrino_info.txt prints.
        # ------------------------------------------------------------------
        event_vertex_records = build_neutrino_vertex_records(
            flatten_mc_tree(result['mc']), clusters_true, input_file_name, evt, event_key,
            x_min=x_min, x_max=x_max, y_min=y_min, y_max=y_max, z_min=z_min, z_max=z_max,
            clusters_true_precut=clusters_true_precut, min_cluster_energy=min_cluster_energy)

        for key, _, _ in TRUE_STAGES:
            ids = true_stage_ids[key]
            job_true_counts[key]['cosmic'] += sum(
                1 for cid in ids if not is_neutrino_by_cid.get(cid, False))
            for record in event_vertex_records:
                # no-cut stage counts every interaction; later stages only those
                # whose cluster is still present
                if key != 'nocut' and record['cluster_id'] not in ids:
                    continue
                if record['vertex_in_volume'] is True:
                    job_true_counts[key]['neutrino_in'] += 1
                else:
                    job_true_counts[key]['neutrino_out'] += 1

        # ------------------------------------------------------------------
        # RECO SIDE: no truth label, total only. No geometry cut -- see header.
        # ------------------------------------------------------------------
        reco_stage_ids = {
            'nocut': set(clusters_all_clu.keys()),
            'beam':  set(clusters_clu_in_beam_window.keys()),
        }
        for key, _, _ in RECO_STAGES:
            job_reco_counts[key]['total'] += len(reco_stage_ids[key])

        job_cluster_type_records.extend(event_cluster_type_records)
        job_vertex_records.extend(event_vertex_records)

        total_events_processed += 1
        n_nu_energy = sum(1 for r in event_vertex_records if r['cluster_id'] in true_stage_ids['energy'])
        print(f"  {event_key}: true clusters {len(true_stage_ids['nocut'])} -> energy {len(true_stage_ids['energy'])}"
              f" -> all cuts {len(clusters_true)}"
              f"   |   mc neutrinos {len(event_vertex_records)} -> energy {n_nu_energy}"
              f"   |   reco {len(reco_stage_ids['nocut'])} -> beam {len(reco_stage_ids['beam'])}", flush=True)

print(f"\n{'='*70}")
print(f"JOB SUMMARY: {total_files_processed} file(s), {total_events_processed} event(s) processed")
print(f"Cluster-type records: {len(job_cluster_type_records)}, neutrino vertex records: {len(job_vertex_records)}")
print(f"{'='*70}")


Output directory: multi_file_plots_charge_light_matching/selection_analysis/20260805_163812


FILE 1/12: Haiwang_files_charge_light_matching_MCP2025C_Fall_production_after_deadareacut/file0
  file0_0: true clusters 222 -> energy 7 -> all cuts 7   |   mc neutrinos 1 -> energy 0   |   reco 15 -> beam 0
  file0_1: true clusters 127 -> energy 6 -> all cuts 6   |   mc neutrinos 1 -> energy 1   |   reco 13 -> beam 1
  file0_2: true clusters 25 -> energy 10 -> all cuts 10   |   mc neutrinos 2 -> energy 2   |   reco 14 -> beam 1
  file0_3: true clusters 184 -> energy 10 -> all cuts 10   |   mc neutrinos 1 -> energy 1   |   reco 17 -> beam 1
  file0_4: true clusters 179 -> energy 14 -> all cuts 14   |   mc neutrinos 2 -> energy 2   |   reco 22 -> beam 2
  file0_5: true clusters 486 -> energy 7 -> all cuts 7   |   mc neutrinos 2 -> energy 1   |   reco 12 -> beam 1
  file0_6: true clusters 40 -> energy 8 -> all cuts 8   |   mc neutrinos 1 -> energy 1   |   reco 10 -> beam 1
  file0_7: true clust

In [23]:
# ============================================================================
# DRAW -- selection flow
# ============================================================================
# Every stage is COUNTED and written to selection_flow_counts.txt; the drawn bars
# are the same set. There is no true-side beam-window block ("in beam window" is
# not a truth quantity -- true clusters carry no flash and no time) and no
# dead-area block on either side (meaningless on reco, and on true it removes no
# whole cluster). See the header markdown for the reasoning.
true_stage_records = [
    {'key': key, 'stage': label, 'geometry': is_geometry, **job_true_counts[key]}
    for key, label, is_geometry in TRUE_STAGES
]
reco_stage_records = [
    {'key': key, 'stage': label, 'geometry': is_geometry, **job_reco_counts[key]}
    for key, label, is_geometry in RECO_STAGES
]

true_plot_records = list(true_stage_records)
reco_plot_records = list(reco_stage_records)

# include_geometry_cuts=False so the filename/title say so honestly -- the records
# passed in already carry no geometry stage, making the drawer's own filter a no-op.
DrawTrueClusterSelectionFlow(true_plot_records, output_dir, "Job Level", "job", "Combined",
                             include_geometry_cuts=False)
DrawRecoClusterSelectionFlow(reco_plot_records, output_dir, "Job Level", "job", "Combined",
                             include_geometry_cuts=False)

table_path = write_selection_flow_table(true_stage_records, reco_stage_records, output_dir,
                                        level_name="Job Level")

print("TRUE selection flow (cosmic clusters | neutrino interactions by vertex volume):")
for r in true_stage_records:
    n_nu = r['neutrino_in'] + r['neutrino_out']
    print(f"  {r['stage']:<32} cosmic={r['cosmic']:>6}  neutrino={n_nu:>5} "
          f"(in-volume {r['neutrino_in']}, out-volume {r['neutrino_out']})")
reco_drawn = ", ".join(r['stage'] for r in reco_plot_records)
print(f"\nRECO cluster selection flow (drawn: {reco_drawn}):")
for r in reco_stage_records:
    print(f"  {r['stage']:<32} total={r['total']:>6}")

print(f"\nPlots written to: {output_dir}")
print(f"Counts table written to: {table_path}")

# ============================================================================
# SUMMARY (job level): configuration, what was processed, the flow counts, and
# the job's start/finish/runtime -- the same JOB RUNTIME section the main
# notebook's job_summary/summary.txt carries, so the two are comparable.
# ============================================================================
job_finish_time = time.time()
job_finish_dt   = datetime.now()
job_runtime     = job_finish_time - job_start_time

summary_lines = []
summary_lines.append("=" * 80)
summary_lines.append("SELECTION ANALYSIS SUMMARY")
summary_lines.append("=" * 80)
summary_lines.append(f"Generated: {job_finish_dt.strftime('%Y-%m-%d %H:%M:%S')}")
summary_lines.append("")
summary_lines.append("Configuration:")
summary_lines.append(f"  Parent directory: {PARENT_DIR}")
summary_lines.append(f"  Files to process:  {files}")
summary_lines.append(f"  Events to process: {events}")
summary_lines.append(f"  target_file: {target_file}, target_event: {target_event}, "
                     f"target_file_range: {target_file_range}")
summary_lines.append(f"  Energy cut: {min_cluster_energy} MeV (applied: {Apply_energy_cutoff})")
summary_lines.append(f"  Dead area cut (true side only): applied: {Apply_deadarea_cut}")
summary_lines.append(f"  Volume (wire-readout sensitive box): "
                     f"x [{x_min:g}, {x_max:g}], y [{y_min:g}, {y_max:g}], z [{z_min:g}, {z_max:g}] cm")
summary_lines.append("")
summary_lines.append(f"Files processed:  {total_files_processed}")
summary_lines.append(f"Events processed: {total_events_processed}")
summary_lines.append("")

summary_lines.append("=" * 80)
summary_lines.append("TRUE SELECTION FLOW")
summary_lines.append("=" * 80)
summary_lines.append("cosmic   = cosmic CLUSTERS surviving the stage")
summary_lines.append("neutrino = true neutrino INTERACTIONS from mc.json, split by vertex volume")
for r in true_stage_records:
    n_nu = r['neutrino_in'] + r['neutrino_out']
    summary_lines.append(f"  {r['stage']:<34} cosmic={r['cosmic']:>7}  neutrino={n_nu:>6} "
                         f"(in-volume {r['neutrino_in']}, out-volume {r['neutrino_out']})")
summary_lines.append("")
summary_lines.append("=" * 80)
summary_lines.append("RECO SELECTION FLOW")
summary_lines.append("=" * 80)
for r in reco_stage_records:
    summary_lines.append(f"  {r['stage']:<34} total={r['total']:>7}")
summary_lines.append("")

summary_lines.append("=" * 80)
summary_lines.append("JOB RUNTIME")
summary_lines.append("=" * 80)
summary_lines.append(f"Job started at:  {datetime.fromtimestamp(job_start_time).strftime('%Y-%m-%d %H:%M:%S')}")
summary_lines.append(f"Job finished at: {job_finish_dt.strftime('%Y-%m-%d %H:%M:%S')}")
summary_lines.append(f"Total job runtime: {timedelta(seconds=int(job_runtime))} ({job_runtime:.1f} seconds)")
summary_lines.append("=" * 80)

with open(output_dir / "summary.txt", "w") as f:
    f.write("\n".join(summary_lines) + "\n")

print(f"\nJob finished at: {job_finish_dt.strftime('%Y-%m-%d %H:%M:%S')} (runtime: {job_runtime:.1f}s)")
print(f"Summary written to: {output_dir / 'summary.txt'}")

TRUE selection flow (cosmic clusters | neutrino interactions by vertex volume):
  No cuts                          cosmic= 20225  neutrino=  223 (in-volume 72, out-volume 151)
  True Deposited energy >= 100 MeV cosmic=   980  neutrino=   72 (in-volume 53, out-volume 19)

RECO cluster selection flow (drawn: No cuts, + Beam window):
  No cuts                          total=  1856
  + Beam window                    total=    88

Plots written to: multi_file_plots_charge_light_matching/selection_analysis/20260805_163812
Counts table written to: multi_file_plots_charge_light_matching/selection_analysis/20260805_163812/selection_flow_counts.txt

Job finished at: 2026-08-05 16:39:01 (runtime: 49.0s)
Summary written to: multi_file_plots_charge_light_matching/selection_analysis/20260805_163812/summary.txt


In [24]:
# ============================================================================
# JOB-LEVEL NEUTRINO SELECTION PLOTS AND INFO FILES
# ============================================================================
# Moved here from Evaluation_ChargeLightMatching_BeforeBeamWindowCut.ipynb, which
# used to draw these at job level alongside its completeness/purity work. They are
# selection-analysis products -- what the truth contains and what the selections
# keep -- not reconstruction-performance measurements, so they belong here.
#
# Same functions, same arguments, same output filenames as before; only the
# calling notebook changed. Written into job_summary/ so the layout matches the
# evaluation notebook's own job_summary/, which still holds everything
# completeness/purity plus cosmic_clusters_by_category (that one stays there --
# it needs cluster_category's track-type classification, which is cluster
# categorisation rather than neutrino selection).
#
# The evaluation notebook STILL draws its own EVENT- and FILE-level copies of the
# six drawers below, deliberately, for debugging individual events and files.
# Only the job-level copies live here.
job_summary_dir = output_dir / "job_summary"
job_summary_dir.mkdir(parents=True, exist_ok=True)

# ----------------------------------------------------------------------------
# TRUE NEUTRINO INFO (job level): one row per true neutrino interaction --
# vertex from mc.json, in/out of the wire-readout sensitive box, the sed-derived
# true cluster energy (the one all cuts use) and mc.json's Etot/Edep alongside it
# for reference only.
# ----------------------------------------------------------------------------
true_neutrino_info_path = write_neutrino_vertex_info(job_vertex_records, job_summary_dir)
if true_neutrino_info_path:
    print(f"True neutrino info written to: {true_neutrino_info_path}")

# Companion file: the interactions the true selections removed, which the plots
# below deliberately exclude (the drawers count surviving clusters only, so they
# agree with labels_aggregated). Kept so those losses can be studied rather than
# vanishing silently.
removed_neutrino_info_path = write_removed_neutrino_info(job_vertex_records, job_summary_dir)
if removed_neutrino_info_path:
    print(f"Removed true neutrino info written to: {removed_neutrino_info_path}")

if b_draw_job_level_plots:
    DrawLabelsAggregated(job_cluster_type_records, job_summary_dir, "Job Level", "job", "Combined",
                         vertex_records=job_vertex_records)
    DrawLabelsByNuIdx(job_cluster_type_records, job_summary_dir, "Job Level", "job", "Combined")
    DrawNeutrinoVertices(job_vertex_records, job_summary_dir, "Job Level", "job", "Combined",
                         x_min=x_min, x_max=x_max, y_min=y_min, y_max=y_max, z_min=z_min, z_max=z_max)
    DrawNeutrinoVolumeCategory(job_vertex_records, job_summary_dir, "Job Level", "job", "Combined")
    DrawNeutrinoFlavor(job_vertex_records, job_summary_dir, "Job Level", "job", "Combined")
    DrawNeutrinoBreakdown(job_vertex_records, job_summary_dir, "Job Level", "job", "Combined",
                          energy_threshold=min_cluster_energy)
    plt.close('all')
    print(f"\nJob-level neutrino selection plots written to: {job_summary_dir}")
else:
    print("\nb_draw_job_level_plots is False -- info files written, plots skipped")

n_with_cluster = sum(1 for r in job_vertex_records if r.get('has_true_cluster'))
n_in_volume    = sum(1 for r in job_vertex_records if r.get('vertex_in_volume') is True)
print(f"\nNeutrino interactions (mc.json):        {len(job_vertex_records)}")
print(f"  vertex inside the sensitive volume:   {n_in_volume}")
print(f"  with a true cluster surviving cuts:   {n_with_cluster}")
print(f"True clusters after all cuts:           {len(job_cluster_type_records)}"
      f" ({sum(1 for r in job_cluster_type_records if r['is_neutrino'])} neutrino)")

True neutrino info written to: multi_file_plots_charge_light_matching/selection_analysis/20260805_163812/job_summary/true_neutrino_info.txt
Removed true neutrino info written to: multi_file_plots_charge_light_matching/selection_analysis/20260805_163812/job_summary/removed_true_neutrino_info.txt

Job-level neutrino selection plots written to: multi_file_plots_charge_light_matching/selection_analysis/20260805_163812/job_summary

Neutrino interactions (mc.json):        223
  vertex inside the sensitive volume:   72
  with a true cluster surviving cuts:   72
True clusters after all cuts:           1052 (72 neutrino)
